# 🏆 ENTRENAMIENTO FINAL - BETO V2_FINAL

## ¿Por qué entrenamos con todos los datos?

En la FASE anterior:
- Validamos que BETO V2 funciona correctamente
- Obtuvimos métricas reales con datos nunca vistos (test set)
- Confirmamos overfitting mínimo (2.16%)

Ahora que el método está validado, entrenamos el **modelo final**
con el dataset completo (train + val + test = 8,457 ejemplos).

**Más datos = modelo más robusto en producción.**

> ⚠️ No evaluamos este modelo con métricas nuevas porque
> ya NO tenemos datos independientes para hacerlo.
> Las métricas oficiales son las del modelo V2 (F1=87.60%).

In [4]:
print("Instalando dependencias...")
!pip install -q transformers datasets evaluate accelerate
print("✅ Dependencias instaladas")

Instalando dependencias...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
✅ Dependencias instaladas


In [5]:
import pandas as pd
import numpy as np
import torch
import os
import json
import time
import shutil
import warnings
warnings.filterwarnings('ignore')

print(f"✅ Imports básicos OK")
print(f"   PyTorch: {torch.__version__}")
print(f"   GPU: {torch.cuda.is_available()}")

✅ Imports básicos OK
   PyTorch: 2.10.0+cu128
   GPU: True


In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import evaluate
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

print("✅ Imports ML OK")

✅ Imports ML OK


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Sin GPU. Ve a Runtime → Change runtime type → GPU")

✅ Dispositivo: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


In [8]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

print(f"Cargando tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer cargado")

Cargando tokenizer: dccuchile/bert-base-spanish-wwm-cased


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

✅ Tokenizer cargado


In [9]:
# Función de tokenización
def tokenize_function(examples):
    return tokenizer(
        examples['texto'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Métricas
accuracy_metric  = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")
f1_metric        = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy':  accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision_metric.compute(predictions=preds, references=labels, average='binary')['precision'],
        'recall':    recall_metric.compute(predictions=preds, references=labels, average='binary')['recall'],
        'f1':        f1_metric.compute(predictions=preds, references=labels, average='binary')['f1']
    }

print("✅ tokenize_function definida")
print("✅ compute_metrics definida")

✅ tokenize_function definida
✅ compute_metrics definida


In [10]:
print("Cargando datasets V2...")

df_train = pd.read_csv('train_v2.csv')
df_val   = pd.read_csv('val_v2.csv')
df_test  = pd.read_csv('test_v2.csv')

print(f"✅ Train:  {len(df_train)} ejemplos")
print(f"✅ Val:    {len(df_val)} ejemplos")
print(f"✅ Test:   {len(df_test)} ejemplos")
print(f"✅ Total:  {len(df_train)+len(df_val)+len(df_test)} ejemplos")

print("Combinando en dataset completo...")

df_completo = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_completo = df_completo.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Dataset completo: {len(df_completo)} ejemplos")
print(f"   Clase 0 (no ofensivo): {(df_completo['label']==0).sum()} ({(df_completo['label']==0).mean()*100:.1f}%)")
print(f"   Clase 1 (ofensivo):    {(df_completo['label']==1).sum()} ({(df_completo['label']==1).mean()*100:.1f}%)")

print(f"\nPrimeras 3 filas:")
print(df_completo.head(3))

Cargando datasets V2...
✅ Train:  5919 ejemplos
✅ Val:    1269 ejemplos
✅ Test:   1269 ejemplos
✅ Total:  8457 ejemplos
Combinando en dataset completo...
✅ Dataset completo: 8457 ejemplos
   Clase 0 (no ofensivo): 4260 (50.4%)
   Clase 1 (ofensivo):    4197 (49.6%)

Primeras 3 filas:
                                               texto  label
0  emilio moro medita ajustar el precio de su vin...      0
1  no sé cómo sigues sin darte cuenta de que eres...      1
2  ¿qué esperas, que te aplaudamos por ser un fra...      1


In [11]:
print("Tokenizando dataset completo...")
print("(Puede tardar ~30 segundos)")

dataset_completo = Dataset.from_pandas(df_completo)
dataset_completo = dataset_completo.map(tokenize_function, batched=True)
dataset_completo = dataset_completo.rename_column("label", "labels")
dataset_completo.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print(f"\n✅ Tokenización completada")
print(f"   Total ejemplos: {len(dataset_completo)}")
print(f"   Shape input_ids: {dataset_completo[0]['input_ids'].shape}")

Tokenizando dataset completo...
(Puede tardar ~30 segundos)


Map:   0%|          | 0/8457 [00:00<?, ? examples/s]


✅ Tokenización completada
   Total ejemplos: 8457
   Shape input_ids: torch.Size([128])


In [15]:
print("Cargando modelo BETO fresco...")

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "No ofensivo", 1: "Ofensivo"},
    label2id={"No ofensivo": 0, "Ofensivo": 1}
)

# Layer Freezing
print("Aplicando Layer Freezing...")

for param in model_final.bert.embeddings.parameters():
    param.requires_grad = False

CAPAS_A_CONGELAR = 8
for i in range(CAPAS_A_CONGELAR):
    for param in model_final.bert.encoder.layer[i].parameters():
        param.requires_grad = False

model_final = model_final.to(device)
model_final.train()

total_params     = sum(p.numel() for p in model_final.parameters())
trainable_params = sum(p.numel() for p in model_final.parameters() if p.requires_grad)

print(f"\n✅ Modelo cargado con Layer Freezing:")
print(f"   Total parámetros:       {total_params:,}")
print(f"   Parámetros entrenables: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"   Capas congeladas:       0-{CAPAS_A_CONGELAR-1} + embeddings")
print(f"   Capas entrenables:      {CAPAS_A_CONGELAR}-11 + clasificador")

Cargando modelo BETO fresco...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Aplicando Layer Freezing...

✅ Modelo cargado con Layer Freezing:
   Total parámetros:       109,852,418
   Parámetros entrenables: 28,943,618 (26.3%)
   Capas congeladas:       0-7 + embeddings
   Capas entrenables:      8-11 + clasificador


In [16]:
training_args_final = TrainingArguments(
    output_dir="./beto-v2-final",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.1,
    warmup_steps=500,
    logging_dir='./logs_final',
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
    fp16=True
)

print("✅ Training Arguments configurados:")
print(f"   Epochs:       {training_args_final.num_train_epochs}")
print(f"   LR:           {training_args_final.learning_rate}")
print(f"   Weight decay: {training_args_final.weight_decay}")
print(f"   Dataset:      {len(dataset_completo)} ejemplos")
print(f"   Sin evaluación (no hay val set separado)")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training Arguments configurados:
   Epochs:       2
   LR:           2e-05
   Weight decay: 0.1
   Dataset:      8457 ejemplos
   Sin evaluación (no hay val set separado)


In [17]:
print("="*70)
print("ENTRENANDO V2_FINAL")
print("="*70)

trainer_final = Trainer(
    model=model_final,
    args=training_args_final,
    train_dataset=dataset_completo,
)

print(f"\n📊 Resumen:")
print(f"   Dataset: {len(dataset_completo)} ejemplos")
print(f"   Epochs:  {training_args_final.num_train_epochs}")
print(f"\n⏰ Tiempo estimado: ~5-8 minutos")
print("🚀 Iniciando...\n")
print("-"*70)

inicio = time.time()
trainer_final.train()
fin = time.time()

print("-"*70)
print(f"\n✅ V2_FINAL entrenado en {(fin-inicio)/60:.1f} minutos")
print("="*70)

ENTRENANDO V2_FINAL

📊 Resumen:
   Dataset: 8457 ejemplos
   Epochs:  2

⏰ Tiempo estimado: ~5-8 minutos
🚀 Iniciando...

----------------------------------------------------------------------


Step,Training Loss
50,0.684248
100,0.578688
150,0.454426
200,0.415522
250,0.394392
300,0.343365
350,0.379539
400,0.308494
450,0.364456
500,0.356645


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

----------------------------------------------------------------------

✅ V2_FINAL entrenado en 1.7 minutos


In [18]:
from datetime import datetime

print("Guardando V2_FINAL...")

os.makedirs('beto_v2_final/modelo', exist_ok=True)
os.makedirs('beto_v2_final/tokenizer', exist_ok=True)

model_final.save_pretrained('beto_v2_final/modelo')
print("✅ Modelo guardado")

tokenizer.save_pretrained('beto_v2_final/tokenizer')
print("✅ Tokenizer guardado")

metricas_finales = {
    "modelo": "BETO V2_FINAL",
    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_total": len(dataset_completo),
    "epochs": training_args_final.num_train_epochs,
    "metricas_oficiales": {
        "nota": "Evaluadas con test set independiente (modelo V2)",
        "accuracy": 0.8834,
        "precision": 0.9273,
        "recall": 0.8302,
        "f1_score": 0.8760,
        "roc_auc": 0.9436,
        "pr_auc": 0.9555,
        "overfitting_gap": 0.0216
    }
}

with open('beto_v2_final/metricas_v2_final.json', 'w', encoding='utf-8') as f:
    json.dump(metricas_finales, f, indent=4, ensure_ascii=False)
print("✅ Métricas guardadas")

print("\n📁 Estructura:")
print("   beto_v2_final/")
print("   ├── modelo/")
print("   ├── tokenizer/")
print("   └── metricas_v2_final.json")

Guardando V2_FINAL...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo guardado
✅ Tokenizer guardado
✅ Métricas guardadas

📁 Estructura:
   beto_v2_final/
   ├── modelo/
   ├── tokenizer/
   └── metricas_v2_final.json


In [19]:
from google.colab import files

print("Creando ZIP...")
shutil.make_archive('beto_v2_final', 'zip', 'beto_v2_final')
size_mb = os.path.getsize('beto_v2_final.zip') / (1024**2)
print(f"✅ ZIP creado: {size_mb:.1f} MB")

print("\nDescargando archivos...")
files.download('beto_v2_final.zip')
files.download('beto_v2_final/metricas_v2_final.json')
print("✅ Descarga iniciada")

Creando ZIP...
✅ ZIP creado: 389.2 MB

Descargando archivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga iniciada
